In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.tree import DecisionTreeClassifier
from sklearn.cluster import KMeans
from sklearn.metrics import accuracy_score, f1_score,confusion_matrix,classification_report
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
df = pd.read_csv(r"C:\Users\Administrator\Desktop\movie_recommendation_system\data\imdb_raw.csv")
df.head()
df.info()
df.isnull().sum()

In [ ]:
df.drop_duplicates(inplace=True)
df['rating'].fillna(df['rating'].mean(), inplace=True)
df['metascore'].fillna(df['metascore'].mean(), inplace=True)
df.dropna(subset=['title', 'genre'], inplace=True)
df.to_csv(r"C:\Users\Administrator\Desktop\movie_recommendation_system\data\imdb_clean.csv", index=False)

In [ ]:
plt.figure(figsize=(6,4))
sns.histplot(df['rating'], bins=10)
plt.title("Movie Rating Distribution")
plt.show()
df['genre'].str.split(',').explode().value_counts().head(10)

In [ ]:
genre_df = df.assign(genre=df['genre'].str.split(',')).explode('genre')
genre_df.groupby('genre')['rating'].mean().sort_values(ascending=False).head(10)

In [ ]:
df['liked'] = (df['rating'] >= df['rating'].median()).astype(int)

In [ ]:
tfidf = TfidfVectorizer(tokenizer=lambda x: x.split(','),token_pattern=None)
genre_tfidf = tfidf.fit_transform(df['genre'])


In [ ]:
X = pd.concat([df[['runtime', 'metascore']],genre_tfidf_df], axis=1)
y = df['liked']


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred_lr))
print("F1 Score:", f1_score(y_test, y_pred_lr))

In [ ]:
cm_lr = confusion_matrix(y_test, y_pred_lr)
plt.figure(figsize=(4,3))
sns.heatmap(cm_lr, annot=True, fmt='d', cmap='Blues')
plt.title("Logistic Regression Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

In [ ]:
dt = DecisionTreeClassifier(max_depth=5)
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred_dt))
print("F1 Score:", f1_score(y_test, y_pred_dt))


In [ ]:
cm_dt = confusion_matrix(y_test, y_pred_dt)
plt.figure(figsize=(4,3))
sns.heatmap(cm_dt, annot=True, fmt='d', cmap='Greens')
plt.title("Decision Tree Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

In [ ]:
kmeans = KMeans(n_clusters=5, random_state=42)
df['cluster'] = kmeans.fit_predict(genre_tfidf)


In [ ]:
def recommend_movies(movie_title, n=5):
    matches = df[df['title'].str.lower() == movie_title.lower()]
    if matches.empty:
        return "Movie not found in dataset"
    cluster = matches['cluster'].iloc[0]
    return (df[df['cluster'] == cluster].sort_values(by='rating', ascending=False)['title'].head(n))


In [ ]:
recommend_movies(df['title'].iloc[0])
recommend_movies("The Shawshank Redemption")

USAGE OF THIS MODEL FOR:
	•	Clear insights into movie trends and user preferences
	•	A working movie preference prediction model
	•	A basic yet effective recommendation system
Logistic Regression performed better than Decision Tree because it was less prone to overfitting and achieved higher accuracy and F1-score.
